# BioSkills Lab — Chapter 9
## Neural Networks and Autoencoders on TCGA
[![BioSkills Lab](https://img.shields.io/badge/BioSkills-Lab-38bdf8)](https://bioskillslab.dev)

> **GPU recommended** — Runtime > Change runtime type > T4 GPU

In [ ]:
!pip install -q torch scikit-learn numpy pandas matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
print(f'GPU: {torch.cuda.is_available()}')

In [ ]:
# Assumes X_train_pca, X_test_pca, y_train, y_test from Chapter 7
# Neural Network Classifier
class CancerClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim//2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim//2, output_dim)
        )
    def forward(self, x):
        return self.network(x)

In [ ]:
n_classes = len(np.unique(y_train))
model = CancerClassifier(X_train_pca.shape[1], 256, n_classes)
criterion = nn.CrossEntropyLoss()
optimiser = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

train_ds = TensorDataset(torch.FloatTensor(X_train_pca), torch.LongTensor(y_train))
loader   = DataLoader(train_ds, batch_size=256, shuffle=True)

train_accs, test_accs = [], []
for epoch in range(50):
    model.train()
    for xb, yb in loader:
        optimiser.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward(); optimiser.step()
    model.eval()
    with torch.no_grad():
        tr_acc = (model(torch.FloatTensor(X_train_pca)).argmax(1) == torch.LongTensor(y_train)).float().mean().item()
        te_acc = (model(torch.FloatTensor(X_test_pca)).argmax(1) == torch.LongTensor(y_test)).float().mean().item()
    train_accs.append(tr_acc); test_accs.append(te_acc)
    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1}: train={tr_acc:.3f}, test={te_acc:.3f}')

In [ ]:
plt.plot(train_accs, label='Train')
plt.plot(test_accs, label='Test')
plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.title('Neural Network Learning Curves'); plt.legend(); plt.show()

In [ ]:
# Autoencoder
class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim,512),nn.ReLU(),nn.Linear(512,256),nn.ReLU(),nn.Linear(256,latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim,256),nn.ReLU(),nn.Linear(256,512),nn.ReLU(),nn.Linear(512,input_dim))
    def forward(self, x):
        z = self.encoder(x); return self.decoder(z), z
    def encode(self, x): return self.encoder(x)

ae = Autoencoder(X_train_pca.shape[1], 32)
ae_opt = optim.Adam(ae.parameters(), lr=1e-3)
ae_loader = DataLoader(TensorDataset(torch.FloatTensor(X_train_pca)), batch_size=256, shuffle=True)

for epoch in range(100):
    ae.train()
    for (xb,) in ae_loader:
        ae_opt.zero_grad()
        xr, _ = ae(xb)
        loss = nn.MSELoss()(xr, xb)
        loss.backward(); ae_opt.step()
    if (epoch+1) % 25 == 0:
        print(f'Epoch {epoch+1}: recon_loss={loss.item():.4f}')

In [ ]:
ae.eval()
with torch.no_grad():
    Z_train = ae.encode(torch.FloatTensor(X_train_pca)).numpy()
    Z_test  = ae.encode(torch.FloatTensor(X_test_pca)).numpy()
lr_ae = LogisticRegression(max_iter=1000)
lr_ae.fit(Z_train, y_train)
print(f'Autoencoder + LR accuracy: {accuracy_score(y_test, lr_ae.predict(Z_test)):.3f}')